In [2]:
from pypdf import PdfReader
import fitz 
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
import time

In [3]:
def request(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")
    return soup

In [4]:
# function to retreive html for sites when requests doesnt work
def scroll(url):
    driver = webdriver.Chrome()
    driver.get(url)
    last_height = driver.execute_script("return document.body.scrollHeight")

    # Scroll to bottom
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        new_height = driver.execute_script("return document.body.scrollHeight")

        # Stop if no new content loaded
        if new_height == last_height: break
            
        last_height = new_height

    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()
    return soup

In [5]:
# Colorado Boating Handbook
doc = fitz.open("PDF Source Files/Colorado_Boating_Handbook.pdf")

all_text = []

for page_num, page in enumerate(doc):

    # text only
    text = page.get_text()
    
    # image metadata
    images = page.get_images(full=True)

    all_text.append({
        "page": page_num + 1,
        "text": text,
        "num_images": len(images)
    })

# pages in this pdf are very short with minimal text as it's a handbook to keep on your boat
clean_text = []
for i in all_text:
    clean_i = i['text'].replace('�', '').replace('ຌ', '').replace('ຈ', '')
    clean_text.append(clean_i)

# length of choosen pages is closer to 10 pages of normal text
keep_pages = [2, 5, 7, 10, 13, 14, 15, 25, 30, 34, 35, 36, 37, 38, 42, 43, 44, 46, 49, 50, 51, 52]

clean_text_keep = [clean_text[i] for i in keep_pages]


with open("Cleaned Txt Files/Colorado_Boating_Handbook.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(clean_text_keep))

In [6]:
# Colorado Kayaking Laws
soup = scroll('https://paddlecamp.com/colorado-kayaking-laws/')

text = soup.find('h1').get_text() + ' \n' + soup.find('div', class_="entry-content").get_text()

with open("Cleaned Txt Files/Colorado_Kayaking_Laws.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [7]:
# Colorado Public Law on Paddleboards
soup = request('https://colorado.public.law/statutes/crs_33-13-110')

body_list = [section.get_text() for section in soup.find('div', id="leaf-statute-body").find_all('section')]
body = ' \n'.join(body_list)

text = soup.find('h1').get_text().strip() + body

with open("Cleaned Txt Files/Colorado_Paddleboard_Laws.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [8]:
# Guide to Paddleboarding in Colorado
soup = request('https://www.yocolorado.com/blogs/news/the-ultimate-guide-to-paddle-boarding-in-colorado')

text = soup.find('div', class_="article").get_text().split('Share\nShare')[0]

with open("Cleaned Txt Files/Colorado_Paddleboard_Guide.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [9]:
# Kayak and Paddleboard Safety Checklist
reader = PdfReader("PDF Source Files/Kayak_Paddle_Board_Checklist.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text()

text = text.replace('  ', ' \n').replace(' 7.', '\n7.').replace(' 11.', '\n11.').replace(' 12.', '\n12.').replace(' 16.', '\n16.')

with open("Cleaned Txt Files/Safety_Checklist.txt", "w", encoding="utf-8") as f:
    f.write(text)

Ignoring wrong pointing object 7 0 (offset 0)


In [10]:
# Best Places to Paddleboard in Colorado 
soup = request('https://www.standuppaddleboarding.com/best-places-to-paddle-board-in-colorado-7538359/')

title = soup.find('h1').get_text()     
intro = soup.find('div', class_="r am g").get_text()

clean_body_list = []

for i in soup.find_all('div', class_="r am g")[2:]:
    first = i.find('h2').get_text() 
    second = i.find('p').get_text() 
    clean_body_list.append(first + ': ' + second)

clean_body = '\n'.join(clean_body_list)
clean_text = title + '\n\n' + intro + '\n\n' + clean_body

with open("Cleaned Txt Files/Best_Paddleboarding_Colorado.txt", "w", encoding="utf-8") as f:
    f.write(clean_text)